## 데이터 로딩 + 확인

* csv 파일 로딩
* 데이터 확인은 다음의 함수로 한다.
    * sample(또는 head), info, describe

In [4]:
# 데이터 로딩

import pandas as pd

df = pd.read_csv('Chemical_Numeric_Data_Quality.csv', encoding='cp949')


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# 가독성 향상을 위해 소수점 출력 포맷을 소수점 넷째자리까지 나오도록 설정
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [7]:
# 데이터 확인
df.sample(3)
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Lot            1000 non-null   int64  
 1   Temp(°C)       870 non-null    float64
 2   Viscosity(cP)  911 non-null    float64
 3   Failure        1000 non-null   object 
 4   Failure code   161 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 39.2+ KB


,Lot,Temp(°C),Viscosity(cP),Failure code
count,1000.0000,870.0000,911.0000,161.0000
mean,2110175932.5680,31.4714,5.5346,3.1056
std,677478.9190,33.0484,11.8892,1.3991
min,2109021818.0000,2.4500,0.7400,1.0000
25%,2109585209.0000,24.3525,1.5700,2.0000
50%,2110189868.0000,25.4300,2.3900,3.0000
75%,2110782208.0000,26.4400,3.2500,4.0000
max,2111289797.0000,435.3500,101.8000,5.0000


## 데이터 정리
* 컬럼 이름에서 데이터의 단위 표기를 제거하기 위해 컬럼 이름을 변경한다.
* 실습 데이터의 FailureCode 데이터가 교재에 나온 데이터 형태와 달라서 그 형태를 맞춘다.

In [8]:
# 컬럼 이름 변경
df.columns = ['Lot', 'Temp', 'Viscosity', 'Failure', 'FailureCode']

# 변경 확인
df.head()

,Lot,Temp,Viscosity,Failure,FailureCode
0,2110262249,24.4400,101.8000,1,2.0000
1,2109975633,23.9500,82.6300,1,NaN
2,2109024936,23.2400,1.5200,1,4.0000
3,2110009104,NaN,81.4900,1,NaN
4,2109169908,23.3300,71.9200,1,NaN


In [9]:
# 실습용 데이터가 교재 실습 데이터와 형태가 달라 맞춰주는 코드
# 지금 이해하지 않아도 괜찮겠다.
df.FailureCode = df.FailureCode.fillna('None').astype(str)
df.FailureCode = df.FailureCode.apply(lambda x: x if x=='None' else str(int(float(x))))

In [10]:
# 데이터 확인 -> 작업 결과 확인
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          1000 non-null   int64  
 1   Temp         870 non-null    float64
 2   Viscosity    911 non-null    float64
 3   Failure      1000 non-null   object 
 4   FailureCode  1000 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 39.2+ KB


,Lot,Temp,Viscosity,Failure,FailureCode
0,2110262249,24.4400,101.8000,1,2
1,2109975633,23.9500,82.6300,1,None
2,2109024936,23.2400,1.5200,1,4
3,2110009104,NaN,81.4900,1,None
4,2109169908,23.3300,71.9200,1,None


## STEP ① 완전성 지표

In [11]:
# 결측치 확인
df.isnull().sum()

# 위 코드는 아래 코드를 단계별로 실행한 것과 같다.
# df.isnull() : df 전체를 대상으로 각 칸이 결측치이면 True,
#               값이 있으면 False 반환
# sum() : isnull()의 결과 dataframe의 각 열 데이터 개수를 센다.
#       : False는 0, True는 1로 계산된다.
#       : 결과적으로는 각 열의 결측치 개수가 된다.

,0
Lot,0
Temp,130
Viscosity,89
Failure,0
FailureCode,0


In [12]:
# 결측치 처리
# df에 데이터 행에 하나라도 결측치가 있으면 행 전체를 삭제
# how = 'any' : df에 데이터 행에 하나라도 결측치가 있으면 행 전체를 삭제
# how = 'all' : df에 데이터 행의 데이터가 모두 결측치면 행을 삭제
df = df.dropna(how='any', axis=0)

# 결과 확인
df.isnull().sum()

,0
Lot,0
Temp,0
Viscosity,0
Failure,0
FailureCode,0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 791 entries, 0 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          791 non-null    int64  
 1   Temp         791 non-null    float64
 2   Viscosity    791 non-null    float64
 3   Failure      791 non-null    object 
 4   FailureCode  791 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 37.1+ KB


## STEP ② 유효성 지표

In [14]:
# 이상치(outlier)
# 온도 20 미만 또는 온도 30 초과
# 또는 점도 3 초과이면 제거 대상으로 선택한다.

# 결과를 outlier에 저장한다.
# 이 줄에서는 아직 df의 행을 삭제하지 않았다.
outlier = df[(df.Temp < 20) | (df.Temp > 30) | (df.Viscosity > 3)]

# (df.Temp < 20) | (df.Temp > 30) | (df.Viscosity > 3)


# 이 코드를 이해하려면 Boolean Indexing을 알아야 한다.
# - df[조건]은 조건이 True인 행만 선택한다.
# - 조건이 여러개 일 때,
#       : 각 비교식은 괄호로 감싼다.
#       : |는 '또는(OR)'으로 조건들을 연결한다.

# 결과 확인
print(outlier)

outlier.sample(10)

            Lot    Temp  Viscosity Failure FailureCode
0    2110262249 24.4400   101.8000       1           2
1    2109975633 23.9500    82.6300       1        None
4    2109169908 23.3300    71.9200       1        None
6    2109032490 54.1100     2.1000      정상        None
7    2109332619 24.3900    71.1400       1        None
..          ...     ...        ...     ...         ...
938  2111185019 26.3600     3.0100      정상        None
947  2111211907 23.6100     3.5500      정상        None
951  2111214194 27.1000    41.0500      불량        None
961  2111224103 24.0500    42.4500       1        None
980  2111256812 44.2200    51.6800      불량        None

[296 rows x 5 columns]


,Lot,Temp,Viscosity,Failure,FailureCode
154,2110083392,25.3700,3.4500,0,None
63,2111066866,23.2800,31.0200,불량,None
871,2111036930,23.2800,3.6500,1,2
227,2110663622,25.3900,3.2300,0,None
155,2109161938,23.9900,3.4500,0,None
147,2110066899,23.4300,3.4700,0,None
188,2109418271,26.2400,3.3800,1,5
626,2109197072,53.4000,1.7400,불량,None
162,2110902700,26.5100,3.4200,0,None
127,2110733544,25.1000,3.5300,0,None


In [15]:
# df에서 outlier에 해당하는 행을 삭제
df = df.drop(outlier.index, axis=0)

In [16]:
# 지금까지 작업한 후 데이터의 정보
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    object 
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 23.2+ KB


## STEP ③ 일관성 지표

In [17]:
# Failure 컬럼은 0 또는 1을 숫자로 가져야하는데,
# 데이터가 문자열이고, "정상" "불량" 이 포함되어 있다.
df.Failure.value_counts()

#[Serise 타입데이터].value_counts()
#[Serise 타입데이터]가 가지고 있는 값의 종류가 각각 몇개 있는지 센다.
#성별이란 컬럼에 있는 값의 종류는 남자, 여자가 있을 때,
#성별.valure_counts() --> 남자인 데이터가 몇 개, 여자인 데이터가 몇 개 센다.


,count
Failure,
0,412
정상,45
1,33
불량,5


In [18]:
# map() 함수: 원본 데이터의 값을 다른 값으로 1:1 매칭해서 변경해주는 함수
# 작업에 필요한 값 : 1. 원본데이터, 2. 무슨 값으로 변경할 건지
# --> Dictionary 데이터 타입으로 표현
# {'정상': 0, '0': 0, '불량': 1, '1': 1}
df.Failure = df.Failure.map({'정상': 0,  #키 '정상'인 데이터 -> 숫자 0으로
                             '0': 0,     #키 '0'인 데이터 -> 숫자 0으로
                             '불량': 1,  #키 '불량'인 데이터 -> 숫자 1으로
                             '1': 1})    #키 '1'인 데이터 -> 숫자 1으로

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    int64  
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 23.2+ KB


In [19]:
# 결과 확인
df.Failure.value_counts()

,count
Failure,
0,457
1,38


### STEP ④ 유일성 지표

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    int64  
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 23.2+ KB


In [21]:
df.Lot.value_counts()

,count
Lot,
2109811675,5
2110753363,4
2110212078,4
2110092816,3
2110496476,3
...,...
2110867774,1
2109727616,1
2110133494,1


In [22]:
# 중복값이 있는지 확인
# 유니크값 Lot
# 중복인 데이터의 개수를 셈
df.Lot.duplicated().sum()

np.int64(40)

In [23]:
# 중복값 삭제
df = df.drop_duplicates(subset=['Lot'], keep='first')
# subset = ['Lot'] : 중복을 판다하는 기준을 'Lot' 컬럼을 기준으로 하겠다.
# keep='first' : 중복 데이터 중에 처음 데이터는 유지하고, 나머지 데이터는 지워라.

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 455 entries, 2 to 975
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          455 non-null    int64  
 1   Temp         455 non-null    float64
 2   Viscosity    455 non-null    float64
 3   Failure      455 non-null    int64  
 4   FailureCode  455 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 21.3+ KB


### STEP ⑤ 정확성 지표

In [25]:
# 데이터 확인
df.info()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 455 entries, 2 to 975
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          455 non-null    int64  
 1   Temp         455 non-null    float64
 2   Viscosity    455 non-null    float64
 3   Failure      455 non-null    int64  
 4   FailureCode  455 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 21.3+ KB


,Lot,Temp,Viscosity,Failure,FailureCode
687,2109361239,26.7800,1.5600,0,None
727,2111084745,24.9800,1.4100,0,None
527,2109347587,26.5400,2.1300,0,None
386,2109714658,23.9900,2.6800,0,None
758,2109076009,26.2100,1.3200,0,None


In [26]:
# 데이터에서 현장을 반영하지 못하는 데이터를 삭제한다.
# CASE1 : 정상인데 FailureCode에 값이 있는 경우
incorrect_condi1_df = df[(df.Failure == 0) & (df.FailureCode != 'None')]
# : Failure가 맞다고 판단해서 처리하고 있음.
# : 정상인게 맞다고 판단 -> FailureCode값을 'None'으로 바꿈
#(df.Failure == 0) & (df.FailureCode != 'None')
df.loc[incorrect_condi1_df.index, 'FailureCode'] = 'None'


# CASE2 : 불량인데(Failure == 1) FailureCode에 값이 없는 경우
incorrect_condi2_df = df[(df.Failure == 1) & (df.FailureCode == 'None')]
df = df.drop(incorrect_condi2_df.index, axis=0)

In [27]:
# 결과 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 449 entries, 2 to 975
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          449 non-null    int64  
 1   Temp         449 non-null    float64
 2   Viscosity    449 non-null    float64
 3   Failure      449 non-null    int64  
 4   FailureCode  449 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 21.0+ KB


In [28]:
import pandas as pd

#<<문제 정의>>
# 사이킷런 라이브러리 import
# 학습데이터 테스트 데이터 분할
# 로지스틱 회귀 모델 생성
# 학습(훈련)
# 모델평가
# - accuracy_score 함수 사용
# - 테스트 데이터와 학습데이터에 대해 구할 것

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#<<데이터 로딩>>


#- 특성 X : 온도, 점도
X = df[['Temp', 'Viscosity']]

#- 타겟 y : 양품/불량 유무
y = df['Failure']

#- 구분: 지도학습 >> 분류
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#1. 로지스틱 회귀 모델을 사용하여 이진 분류 모델을 학습시켜보세요.
lr = LogisticRegression()
lr.fit(train_X, train_y)
print("\n 훈련완료")



#2. 학습 후 모델의 성능을 평가하세요. (정확도 사용)
# 테스트 데이터 예측값과 정확도
y_pred_train = lr.predict(train_X)
accuracy_train = accuracy_score(train_y, y_pred_train)
print("TRAIN Accuracy:", accuracy_train)

# 훈련 데이터 예측값과 정확도
y_pred = lr.predict(test_X)
accuracy = accuracy_score(test_y, y_pred)
print("TEST Accuracy:", accuracy)


#3. 과적합/과소적합 유무를 확인하세요.
#과소적합


 훈련완료
TRAIN Accuracy: 0.9387186629526463
TEST Accuracy: 0.9333333333333333


In [29]:
# 분할하기 전 데이터 y의 분포 (정상과 불량의 개수)
print(y.value_counts())

# 분할 후 훈련데이터 y의 분포 (정상과 불량의 개수)
print(train_y.value_counts())

# 분할 후 테스트데이터 y의 분포 (정상과 불량의 개수)
print(test_y.value_counts())

# 전형적인 불균형 데이터


Failure
0    421
1     28
Name: count, dtype: int64
Failure
0    337
1     22
Name: count, dtype: int64
Failure
0    84
1     6
Name: count, dtype: int64


In [30]:
from sklearn.metrics import confusion_matrix

